20250328

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# from C_0B_eval import *
from C_0X_defs import *
from scipy.stats import sem, ttest_ind
from misc_recorder import ListRecorder
import pandas as pd

In [2]:
ts = "0611193546" # this timestamp does not contain run number
train_name = "C_0T"
# ts = "0912014617"
# train_name = "C_0Tf"
res_save_dir = os.path.join(model_save_, f"eval-{train_name}-{ts}")

results = []
for model_type in ['recon4-phi', 'recon8-phi', 'recon16-phi', 'recon32-phi']: 
    for model_condition in ['b']: 
        for run in range(1, 6): 
            res_read_dir = os.path.join(model_save_, f"{train_name}-{ts}-{run}", f"{model_type}", f"{model_condition}")
            valid_recon_losses = ListRecorder(os.path.join(res_read_dir, "valid.recon.loss"))
            valid_recon_losses.read()
            data = valid_recon_losses.get()
            dimension = 0
            if model_type == 'recon4-phi':
                dimension = 4
            elif model_type == 'recon8-phi': 
                dimension = 8
            elif model_type == 'recon16-phi': 
                dimension = 16
            elif model_type == 'recon32-phi': 
                dimension = 32
            for epoch in range(len(data[:])): 
                recon_loss = data[epoch]
                datadict = {'dimension': dimension, 'model_condition': model_condition, 'run': run, 'epoch': epoch, 'recon_loss': recon_loss}
                results.append(datadict)

In [3]:
resultsdf = pd.DataFrame(results)

In [4]:
resultsdf

,dimension,model_condition,run,epoch,recon_loss
0,4,b,1,0,0.788840
1,4,b,1,1,0.758296
2,4,b,1,2,0.778342
3,4,b,1,3,0.685673
4,4,b,1,4,0.677608
...,...,...,...,...,...
1995,32,b,5,95,0.069654
1996,32,b,5,96,0.068620
1997,32,b,5,97,0.067005
1998,32,b,5,98,0.067192


In [5]:
recon_losses_dict = {}

for d in resultsdf['dimension'].unique(): 
    df = resultsdf[resultsdf['dimension'] == d]

    # Sort the DataFrame by 'epoch'
    df_sorted = df.sort_values(by='epoch')

    # Get the unique run values
    runs = df_sorted['run'].unique()

    # Determine the number of runs and epochs
    num_runs = len(runs)
    num_epochs = df_sorted['epoch'].nunique()

    # Initialize a NumPy array with the shape (run, epoch)
    recon_loss_array = np.empty((num_runs, num_epochs))

    # Populate the NumPy array with recon_loss values
    for i, run in enumerate(runs):
        run_data = df_sorted[df_sorted['run'] == run]
        recon_loss_array[i] = run_data['recon_loss'].values

    recon_losses_dict[d] = recon_loss_array.mean(axis=0, keepdims=True)

In [6]:
from C_0Tf_n_integrate_abx_pph import plot_many

In [7]:
dimensions = [4, 8, 16, 32]
plot_many([recon_losses_dict[d] for d in dimensions], 
          [f"dim = {d}" for d in dimensions], 
          "recon_loss.png", 
          plot_label_dict={"xlabel": "Epoch", "ylabel": "Reconstruction Loss", "title": "Validation Set Learning Curve"})

/home/ldlmdl/anaconda3/envs/wavln/lib/python3.11/site-packages/numpy/core/_methods.py:269: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/ldlmdl/anaconda3/envs/wavln/lib/python3.11/site-packages/numpy/core/_methods.py:258: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


In [34]:
# Perform the groupby and calculate the mean for the remaining column
grouped_df = resultsdf.groupby(["dimension", "model_condition"])["recon_loss"].mean().reset_index()

In [8]:
grouped_df

,dimension,model_condition,recon_loss
0,4,b,0.401569
1,4,u,0.369832
2,8,b,0.222804
3,8,u,0.216290
4,16,b,0.125158
5,16,u,0.119112
6,32,b,0.078389
7,32,u,0.075468


In [9]:
gbres = grouped_df[grouped_df['model_condition'] == 'b']
gures = grouped_df[grouped_df['model_condition'] == 'u']

In [20]:
outstr = ""
for dim in [4, 8, 16, 32]: 
    r = gbres[gbres['dimension'] == dim]['recon_loss'].values[0]
    outstr += "\overline{\epsilon}_{%d} = %.4f, " % (dim, r)

In [21]:
print(outstr)

\overline{\epsilon}_{4} = 0.4016, \overline{\epsilon}_{8} = 0.2228, \overline{\epsilon}_{16} = 0.1252, \overline{\epsilon}_{32} = 0.0784, 


In [22]:
outstr = ""
for dim in [4, 8, 16, 32]: 
    r = gures[gures['dimension'] == dim]['recon_loss'].values[0]
    outstr += "\overline{\epsilon}_{%d} = %.4f, " % (dim, r)

In [23]:
print(outstr)

\overline{\epsilon}_{4} = 0.3698, \overline{\epsilon}_{8} = 0.2163, \overline{\epsilon}_{16} = 0.1191, \overline{\epsilon}_{32} = 0.0755, 


In [19]:
bres = resultsdf[resultsdf['model_condition'] == 'b']
ures = resultsdf[resultsdf['model_condition'] == 'u']

In [21]:
bres["recon_loss"].corr(bres["dimension"]), ures["recon_loss"].corr(ures["dimension"])

(-0.8299008887209766, -0.8565714128555049)